# Phase 1: Dataset Generation & Exploration

This notebook verifies and explores the 4 primary synthetic datasets generated according to the project blueprint:
1. **`banks.csv`** (10 Banks)
2. **`atms.csv`** (500 ATMs across 20 districts in Uttar Pradesh)
3. **`complaints.csv`** (10,000 complaints with realistic spatial/temporal distributions)
4. **`transactions.csv`** (30,000 transactions featuring realistic fraud bursts & background contrast)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = os.path.join("..", "data", "raw")

df_banks = pd.read_csv(os.path.join(DATA_DIR, "banks.csv"))
df_atms = pd.read_csv(os.path.join(DATA_DIR, "atms.csv"))
df_complaints = pd.read_csv(os.path.join(DATA_DIR, "complaints.csv"))
df_transactions = pd.read_csv(os.path.join(DATA_DIR, "transactions.csv"))

print(f"Banks:        {df_banks.shape}")
print(f"ATMs:         {df_atms.shape}")
print(f"Complaints:   {df_complaints.shape}")
print(f"Transactions: {df_transactions.shape}")

## 1. Inspecting ATMs & Spatial Distribution

In [ ]:
display(df_atms.head())

# ATM count by District
plt.figure(figsize=(12, 5))
df_atms["district"].value_counts().plot(kind="bar", color="royalblue")
plt.title("ATM Distribution Across Uttar Pradesh Districts")
plt.xlabel("District")
plt.ylabel("Number of ATMs")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Analyzing Complaints Data

In [ ]:
display(df_complaints.head())

# Complaint Crime Category Distribution
plt.figure(figsize=(10, 4))
sns.countplot(data=df_complaints, y="crime_category", order=df_complaints["crime_category"].value_counts().index, palette="viridis")
plt.title("Complaint Volume by Crime Category")
plt.xlabel("Complaint Count")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

## 3. Analyzing Transactions & Fraud Clusters

Checking normal activity vs fraud bursts, withdrawal amounts, and hourly rhythms.

In [ ]:
display(df_transactions.head())

df_transactions["hour"] = pd.to_datetime(df_transactions["transaction_time"]).dt.hour

plt.figure(figsize=(12, 4))
sns.histplot(data=df_transactions, x="hour", hue="risk_label", multiple="stack", bins=24, palette={0: "skyblue", 1: "crimson"})
plt.title("Hourly Transaction Distribution: Normal (0) vs Fraud Bursts (1)")
plt.xlabel("Hour of Day (0 - 23)")
plt.ylabel("Transaction Count")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Withdrawal Amount Comparison

Examining the contrast between typical citizen withdrawals vs high-velocity fraud spikes.

In [ ]:
withdrawals = df_transactions[df_transactions["transaction_type"] == "WITHDRAWAL"]

print("Mean Withdrawal (Normal): ₹", round(withdrawals[withdrawals["risk_label"] == 0]["amount"].mean(), 2))
print("Mean Withdrawal (Fraud):  ₹", round(withdrawals[withdrawals["risk_label"] == 1]["amount"].mean(), 2))

plt.figure(figsize=(10, 4))
sns.boxplot(data=withdrawals, x="risk_label", y="amount", palette={0: "lightgreen", 1: "salmon"})
plt.title("Withdrawal Amount Distribution: Normal (0) vs Fraud Bursts (1)")
plt.xlabel("Risk Label (0 = Normal, 1 = Fraud / Incident)")
plt.ylabel("Amount (₹)")
plt.tight_layout()
plt.show()

## 5. Acceptance Criteria Checklist Verification

In [ ]:
checks = [
    ("10 Banks Generated", len(df_banks) == 10),
    ("500 ATMs Generated", len(df_atms) == 500),
    ("10,000 Complaints Generated", len(df_complaints) == 10000),
    ("30,000 Transactions Generated", len(df_transactions) == 30000),
    ("No Duplicate ATM IDs", df_atms["atm_id"].is_unique),
    ("No Duplicate Transaction IDs", df_transactions["transaction_id"].is_unique),
    ("Valid Foreign Keys (Bank ID in ATMs)", set(df_atms["bank_id"]).issubset(set(df_banks["bank_id"]))),
    ("Valid Foreign Keys (ATM ID in Txns)", set(df_transactions["atm_id"]).issubset(set(df_atms["atm_id"]))),
    ("Fraud Clusters Linked to Complaints", (df_transactions["complaint_id"].fillna("") != "").sum() > 0),
    ("Synthetic Safety (Tokens Only)", df_transactions["account_id"].str.startswith("ACC").all())
]

for desc, passed in checks:
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} - {desc}")